In [10]:
from pathlib import Path
import sys

# Add the project root to Python's search path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.portfolio import generate_portfolio
    
from src.mortality import(    
    load_ssa_tables,
    prepare_life_table,
    combine_life_tables,
    merge_expected_mortality,
)

Generate a simulated Portfolio with issue age, gender, policy duration and attained age

In [11]:
#Generate the portfolio from ssa tables
portfolio = generate_portfolio()


display(portfolio.head())

portfolio.describe()

,policy_id,issue_age,gender,policy_duration,attained_age
0,1,44,Male,28,72
1,2,27,Male,30,57
2,3,63,Female,25,88
3,4,42,Female,23,65
4,5,47,Female,10,57


,policy_id,issue_age,policy_duration,attained_age
count,10000.00000,10000.00000,10000.000000,10000.000000
mean,5000.50000,45.19820,15.535400,60.733600
std,2886.89568,14.76564,8.661954,16.915107
min,1.00000,20.00000,1.000000,21.000000
25%,2500.75000,33.00000,8.000000,48.000000
50%,5000.50000,45.00000,16.000000,61.000000
75%,7500.25000,58.00000,23.000000,74.000000
max,10000.00000,70.00000,30.000000,100.000000


Load SSA tables and prepare them for merging with the portfolio

In [12]:
female_raw, male_raw = load_ssa_tables()

female = prepare_life_table(female_raw)

male = prepare_life_table(male_raw)

life_table = combine_life_tables(
    male,
    female
)

display(life_table.head())

,attained_age,qx,gender
0,0,0.004907,Female
1,1,0.000316,Female
2,2,0.000196,Female
3,3,0.000160,Female
4,4,0.000129,Female


Assign expected mortality rates to the portfolio using age and sex

In [13]:
portfolio = merge_expected_mortality(
    portfolio,
    life_table
)

display(portfolio.head())

,policy_id,issue_age,gender,policy_duration,attained_age,qx,expected_deaths
0,1,44,Male,28,72,0.030438,0.030438
1,2,27,Male,30,57,0.010687,0.010687
2,3,63,Female,25,88,0.117435,0.117435
3,4,42,Female,23,65,0.011265,0.011265
4,5,47,Female,10,57,0.006227,0.006227


Save the dataset to be used later

In [14]:
portfolio.to_csv(
    "../data/processed/portfolio_with_qx.csv",
    index=False
)